In [1]:
# Nach dieser Zelle: Runtime -> Restart session
%pip install -qU \
    langchain==0.3.25 \
    langchain-openai==0.3.16 \
    langchain-core==0.3.83 \
    langchain-community==0.3.24 \
    langchain-huggingface \
    langsmith \
    datasets \
    pandas \
    huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47

# Lab | Summarization evaluation using LangSmith
Let's revisit your capstone project 2? Well, sort of. Pick diffierent sets of data and re-run this notebook. Maybe parts of the dataset you used in your last project week. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications using LangSmith.

What did you learn? - Let's discuss that in class

## LangSmith - LangChain evaluation

In [56]:
from dotenv import load_dotenv, find_dotenv
import os
from google.colab import userdata

_ = load_dotenv(find_dotenv())

# Try to get API keys from environment variables first, then from Colab's userdata
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY') or userdata.get('OPENAI_API_KEY')
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY") or userdata.get('LANGCHAIN_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN") or userdata.get('HF_TOKEN') # Changed to HF_TOKEN and removed 'None'

# Ensure keys are also set in os.environ for libraries that read directly from it
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
if LANGCHAIN_API_KEY:
    os.environ['LANGCHAIN_API_KEY'] = LANGCHAIN_API_KEY
if HUGGINGFACEHUB_API_TOKEN:
    os.environ['HUGGINGFACEHUB_API_TOKEN'] = HUGGINGFACEHUB_API_TOKEN

In [40]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://eu.api.smith.langchain.com"  # EU region
os.environ["LANGCHAIN_PROJECT"] = "langsmith_max-test"


In [41]:
# Importing Client from Langsmith
from langsmith import Client
client = Client(
    api_key=LANGCHAIN_API_KEY,
    api_url="https://eu.api.smith.langchain.com"
)


### Create Dataset


In [42]:
from datasets import load_dataset
cnn_dataset = load_dataset("cnn_dailymail", "3.0.0")


In [43]:
def add_prefix(example):
    return {
        **example,
        "article": f"Summarize this news:\n{example['article']}"
    }

#cnn_dataset = cnn_dataset.map(add_prefix)

In [44]:
cnn_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [45]:
cnn_dataset['train'][0]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [46]:
#Get just a few news to test
MAX_NEWS=10
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS)).map(add_prefix)

sample_cnn

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 10
})

The dataset contains three columns: article, highlights, and id. To use LangSmith, we need to create a dataset in LangSmith format.

LangSmith expects a prompt and a result. To achieve this, we will transform the article into a prompt by adding the prefix: "Summarize this news." As a result, we will use the content of highlights, which represents the summaries created by humans.

In [47]:
print(sample_cnn[0])

{'article': 'Summarize this news:\n(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki,

Now We have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith with this information.
### Create the Dataset in Langsmith

The dataset in LangSmith is composed of an input, which is the prompt passed to the model for evaluation, and an output, which should contain what we expect the model to return.

In [48]:
import datetime

In [49]:
import uuid
input_key=['article']
output_key=['highlights']

NAME_DATASET=f"Summarize_dataset_{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

In [50]:
from google.colab import userdata
userdata.get('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGSMITH_API_KEY').strip()

In [51]:
from langsmith import Client
client = Client(
    api_key=os.environ["LANGCHAIN_API_KEY"],
    api_url="https://eu.api.smith.langchain.com"
)

# Creates the dataset in LangSmith
dataset = client.create_dataset(
    dataset_name=NAME_DATASET,
    description="Test Embedding distance between model summarizations",
)

# Examples einzeln hochladen
examples = [
    {
        "inputs": {"article": row["article"]},
        "outputs": {"highlights": row["highlights"]},
    }
    for row in sample_cnn
]

client.create_examples(dataset_id=dataset.id, examples=examples)
print(f"Dataset '{NAME_DATASET}' mit {len(examples)} Beispielen erstellt.")

Dataset 'Summarize_dataset_2026-04-07 15:16:44' mit 10 Beispielen erstellt.


In this image, we can see an example from the dataset once it's been registered in LangSmith.

In the Input column, there is the prompt to be sent, while in the Output column, the expected output is stored.

When performing the comparison, the model will be given the prompt, and the Cosine distance between its response and the one stored in the sample dataset will be calculated.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Dataset.jpg?raw=true">

### Recovering Models From Hugging Face
Let's retrieve both models from HuggingFace. A base T5 model and a model that has been fine-tuned using the training portion of this same dataset to generate summaries.

In [70]:
from langchain_huggingface import HuggingFaceEndpoint

In [55]:
import os
from google.colab import userdata

HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN") or userdata.get('HF_TOKEN') # Removed 'None' from userdata.get

if HUGGINGFACEHUB_API_TOKEN:
    os.environ['HUGGINGFACEHUB_API_TOKEN'] = HUGGINGFACEHUB_API_TOKEN
    print("HUGGINGFACEHUB_API_TOKEN wurde erfolgreich geladen.")
else:
    print("HUGGINGFACEHUB_API_TOKEN konnte nicht geladen werden. Bitte stellen Sie sicher, dass er in den Colab-Geheimnissen (als 'HF_TOKEN') oder Umgebungsvariablen gesetzt ist.")

HUGGINGFACEHUB_API_TOKEN wurde erfolgreich geladen.


In [71]:
from langchain_huggingface import HuggingFaceEndpoint

summarizer_base = HuggingFaceEndpoint(
    repo_id="t5-base",
    task="summarization",
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
    temperature=0.1,
    max_new_tokens=180
)

In [73]:
summarizer_finetuned = HuggingFaceEndpoint(
    repo_id="flax-community/t5-base-cnn-dm",
    task="summarization",
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
    temperature=0.1,
    max_new_tokens=180
)

## Defining Evaluator
The first step is to define an evaluator, where we specify the variables we want to evaluate. In our case, I have chosen to measure only the "embedding_distance."

I've left the "string_distance" as a comment in case you want to conduct a test with two evaluations instead of one.


In [74]:
from langchain.smith import RunEvalConfig, run_on_dataset

In [75]:
#We are using just one of the multiple evaluator avaiable on LangSmith.

evaluation_config = RunEvalConfig(
    evaluators=[
        "embedding_distance",
        #"string_distance"
    ],
)



### Running Evaluator
With the same configuration, we can launch two evaluations on the same dataset. One for each of the chosen models.

In [76]:
project_name = f"T5-BASE {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

base_t5_results = run_on_dataset(
    client=client,
    project_name=project_name,
    dataset_name=NAME_DATASET,
    llm_or_chain_factory=summarizer_base,
    evaluation=evaluation_config,
)

View the evaluation results for project 'T5-BASE 2026-04-07 15:22:04' at:
https://eu.smith.langchain.com/o/0e9c6257-f52e-4293-86a2-5f7ff419916b/datasets/159424d3-7194-4273-8414-d5b1bcde9d07/compare?selectedSessions=d91f542d-878b-4175-b395-0675a06854d3

View all tests for Dataset Summarize_dataset_2026-04-07 15:16:44 at:
https://eu.smith.langchain.com/o/0e9c6257-f52e-4293-86a2-5f7ff419916b/datasets/159424d3-7194-4273-8414-d5b1bcde9d07
[>                                                 ] 0/10

Error Type: ValueError, Message: Model 't5-base' doesn't support task 'text-generation'. Supported tasks: 'translation', got: 'text-generation'
Error Type: ValueError, Message: Model 't5-base' doesn't support task 'text-generation'. Supported tasks: 'translation', got: 'text-generation'
Error Type: ValueError, Message: Model 't5-base' doesn't support task 'text-generation'. Supported tasks: 'translation', got: 'text-generation'


[-------------->                                   ] 3/10

Error Type: ValueError, Message: Model 't5-base' doesn't support task 'text-generation'. Supported tasks: 'translation', got: 'text-generation'
Error Type: ValueError, Message: Model 't5-base' doesn't support task 'text-generation'. Supported tasks: 'translation', got: 'text-generation'


[------------------------>                         ] 5/10

Error Type: ValueError, Message: Model 't5-base' doesn't support task 'text-generation'. Supported tasks: 'translation', got: 'text-generation'
Error Type: ValueError, Message: Model 't5-base' doesn't support task 'text-generation'. Supported tasks: 'translation', got: 'text-generation'
Error Type: ValueError, Message: Model 't5-base' doesn't support task 'text-generation'. Supported tasks: 'translation', got: 'text-generation'


[--------------------------------------->          ] 8/10

Error Type: ValueError, Message: Model 't5-base' doesn't support task 'text-generation'. Supported tasks: 'translation', got: 'text-generation'
Error Type: ValueError, Message: Model 't5-base' doesn't support task 'text-generation'. Supported tasks: 'translation', got: 'text-generation'


[------------------------------------------------->] 10/10

In [77]:
#Ignore the error shown below
project_name = f"T5-FineTuned {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

finetuned_t5_results = run_on_dataset(
    client=client,
    project_name=project_name,
    dataset_name=NAME_DATASET,
    llm_or_chain_factory=summarizer_finetuned,
    evaluation=evaluation_config,
)

View the evaluation results for project 'T5-FineTuned 2026-04-07 15:22:18' at:
https://eu.smith.langchain.com/o/0e9c6257-f52e-4293-86a2-5f7ff419916b/datasets/159424d3-7194-4273-8414-d5b1bcde9d07/compare?selectedSessions=e3a916dd-3d11-4209-8abe-66e8d17f5b7e

View all tests for Dataset Summarize_dataset_2026-04-07 15:16:44 at:
https://eu.smith.langchain.com/o/0e9c6257-f52e-4293-86a2-5f7ff419916b/datasets/159424d3-7194-4273-8414-d5b1bcde9d07
[>                                                 ] 0/10

Error Type: StopIteration, Message: 
Error Type: StopIteration, Message: 


[--------->                                        ] 2/10

Error Type: StopIteration, Message: 
Error Type: StopIteration, Message: 
Error Type: StopIteration, Message: 


[------------------------>                         ] 5/10

Error Type: StopIteration, Message: 


... (3 outputs left out)

Error Type: StopIteration, Message: 


[-------------------------------------------->     ] 9/10

Error Type: StopIteration, Message: 


[------------------------------------------------->] 10/10

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Tests.jpg?raw=true">

In the image below you can see the comparision between two tests.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareTestst.jpg?raw=true">

Well, since it has been so straightforward, why don't we try to make the comparison with an OpenAI model?

In [78]:
from langchain_openai import OpenAI
open_aillm=OpenAI(temperature=0.0)

In [79]:
project_name = f"OpenAI {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

finetuned_t5_results = run_on_dataset(
    client=client,
    project_name=project_name,
    dataset_name=NAME_DATASET,
    llm_or_chain_factory=open_aillm,
    evaluation=evaluation_config,
)

View the evaluation results for project 'OpenAI 2026-04-07 15:22:31' at:
https://eu.smith.langchain.com/o/0e9c6257-f52e-4293-86a2-5f7ff419916b/datasets/159424d3-7194-4273-8414-d5b1bcde9d07/compare?selectedSessions=f38fd8ba-1f5c-4251-9b71-0c42f08693d4

View all tests for Dataset Summarize_dataset_2026-04-07 15:16:44 at:
https://eu.smith.langchain.com/o/0e9c6257-f52e-4293-86a2-5f7ff419916b/datasets/159424d3-7194-4273-8414-d5b1bcde9d07
[------------------------------------------------->] 10/10

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareOpenAI_HF.jpg?raw=true">

The experiment with the OpenAI model has yielded the best results. But, be aware! As we can see, there is a cost involved since we are using an API, and it needs to be paid for.

Another crucial piece of information is that we can view performance data for the models. This data could also be useful for minimally evaluating our inference server.